# Visualization of trained models

In [ ]:
import sys
import os
import warnings

import numpy as np
import torch
import pandas as pd

sys.path.insert(0, os.path.abspath(".."))

warnings.filterwarnings("ignore")

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SLURRIES = ["G50", "G45", "G40", "G40+IPA"]

In [ ]:
X = pd.read_csv("../../_temp/v0/X.csv", index_col=[0, 1, 2])
y = pd.read_csv("../../_temp/v0/y.csv")
Xpred = pd.read_csv("../../_temp/v0/Xpred_1D.csv", index_col=[0, 1, 2])

In [ ]:
columns = ["slurry", "cosine_of_contact_angle"]
cos_thetas = X["cosine_of_contact_angle"].reset_index()[columns]
slurry_map = cos_thetas.drop_duplicates().set_index("cosine_of_contact_angle")["slurry"]

slurries = Xpred["cosine_of_contact_angle"].map(slurry_map)
unique_slurries = [s for s in SLURRIES if s in slurries.unique()]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.ticker import ScalarFormatter

Rgt_pred = Xpred["gap_to_thickness_ratio"].unique()
Cas = Xpred["capillary_number"].unique()

Cas_sorted = np.sort(Cas)
cmap = plt.get_cmap("viridis", len(Cas_sorted))
norm = mcolors.BoundaryNorm(
    np.concatenate(
        [
            [Cas_sorted[0] * 0.9],
            (Cas_sorted[:-1] + Cas_sorted[1:]) / 2,
            [Cas_sorted[-1] * 1.1],
        ]
    ),
    ncolors=len(Cas_sorted),
)

## Prior mean

In [ ]:
H_mean = pd.read_csv("../../_temp/v0/H.prior_mean.Xpred_1D.csv").values.flatten()
phi_mean = pd.read_csv("../../_temp/v0/phi.prior_mean.Xpred_1D.csv").values.flatten()

In [ ]:
fig, axes = plt.subplots(1, len(slurry_map), sharex=True, sharey="row")

for slurry, ax in zip(unique_slurries, axes):
    ok = X.index.get_level_values("slurry") == slurry
    this_X = X[ok]
    this_H = y[ok]["H"].values

    ok_pred = slurries == slurry
    this_Xpred = Xpred[ok_pred]
    this_mean = H_mean[ok_pred]

    for ca in this_X["capillary_number"].unique():
        ok = this_X["capillary_number"] == ca
        ok_pred = this_Xpred["capillary_number"] == ca

        ax.scatter(
            this_X[ok]["gap_to_thickness_ratio"],
            this_H[ok],
            color=cmap(norm(ca)),
        )

        ax.plot(
            this_Xpred[ok_pred]["gap_to_thickness_ratio"],
            this_mean[ok_pred],
            color=cmap(norm(ca)),
        )

    (cos_theta,) = this_X["cosine_of_contact_angle"].unique()
    ax.set_title(f"Cos θ={cos_theta:.2f}")

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(
    sm, ax=axes, orientation="horizontal", location="top", pad=0.2, aspect=30
)
cbar.set_label("Ca")
quartile_vals = np.quantile(Cas_sorted, [0, 0.25, 0.5, 0.75, 1.0])
nearest_cas = [Cas_sorted[np.argmin(np.abs(Cas_sorted - q))] for q in quartile_vals]
cbar.set_ticks([round(ca, 3) for ca in nearest_cas])
cbar.ax.xaxis.set_major_formatter(ScalarFormatter())

fig.supxlabel("Rgt")
fig.supylabel("H")
fig.suptitle("Prior mean")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(slurry_map), sharex=True, sharey="row")

for slurry, ax in zip(unique_slurries, axes):
    ok = X.index.get_level_values("slurry") == slurry
    this_X = X[ok]
    this_phi = y[ok]["phi"].values

    ok_pred = slurries == slurry
    this_Xpred = Xpred[ok_pred]
    this_mean = phi_mean[ok_pred]

    for ca in this_X["capillary_number"].unique():
        ok = this_X["capillary_number"] == ca
        ok_pred = this_Xpred["capillary_number"] == ca

        ax.scatter(
            this_X[ok]["gap_to_thickness_ratio"],
            this_phi[ok],
            color=cmap(norm(ca)),
        )

        ax.plot(
            this_Xpred[ok_pred]["gap_to_thickness_ratio"],
            this_mean[ok_pred],
            color=cmap(norm(ca)),
        )

    (cos_theta,) = this_X["cosine_of_contact_angle"].unique()
    ax.set_title(f"Cos θ={cos_theta:.2f}")

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(
    sm, ax=axes, orientation="horizontal", location="top", pad=0.2, aspect=30
)
cbar.set_label("Ca")
quartile_vals = np.quantile(Cas_sorted, [0, 0.25, 0.5, 0.75, 1.0])
nearest_cas = [Cas_sorted[np.argmin(np.abs(Cas_sorted - q))] for q in quartile_vals]
cbar.set_ticks([round(ca, 3) for ca in nearest_cas])
cbar.ax.xaxis.set_major_formatter(ScalarFormatter())

fig.supxlabel("Rgt")
fig.supylabel("phi")
fig.suptitle("Prior mean")
plt.show()

## GPR

In [ ]:
H_gpr = pd.read_csv("../../_temp/v0/H.mean.Xpred_1D.csv")
phi_gpr = pd.read_csv("../../_temp/v0/phi.mean.Xpred_1D.csv")

In [ ]:
fig, axes = plt.subplots(1, len(slurry_map), sharex=True, sharey="row")

for slurry, ax in zip(unique_slurries, axes):
    ok = X.index.get_level_values("slurry") == slurry
    this_X = X[ok]
    this_H = y[ok]["H"].values

    ok_pred = slurries == slurry
    this_Xpred = Xpred[ok_pred]
    this_gpr = H_gpr[ok_pred.values]

    for ca in this_X["capillary_number"].unique():
        ok = this_X["capillary_number"] == ca
        ok_pred = this_Xpred["capillary_number"] == ca

        ax.scatter(
            this_X[ok]["gap_to_thickness_ratio"],
            this_H[ok],
            color=cmap(norm(ca)),
        )

        ax.fill_between(
            this_Xpred[ok_pred]["gap_to_thickness_ratio"],
            this_gpr[ok_pred.values]["H_mean"] - this_gpr[ok_pred.values]["H_std"],
            this_gpr[ok_pred.values]["H_mean"] + this_gpr[ok_pred.values]["H_std"],
            color=cmap(norm(ca)),
            alpha=0.2,
        )

    (cos_theta,) = this_X["cosine_of_contact_angle"].unique()
    ax.set_title(f"Cos θ={cos_theta:.2f}")

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(
    sm, ax=axes, orientation="horizontal", location="top", pad=0.2, aspect=30
)
cbar.set_label("Ca")
quartile_vals = np.quantile(Cas_sorted, [0, 0.25, 0.5, 0.75, 1.0])
nearest_cas = [Cas_sorted[np.argmin(np.abs(Cas_sorted - q))] for q in quartile_vals]
cbar.set_ticks([round(ca, 3) for ca in nearest_cas])
cbar.ax.xaxis.set_major_formatter(ScalarFormatter())

fig.supxlabel("Rgt")
fig.supylabel("H")
fig.suptitle("Posterior distribution of mean")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(slurry_map), sharex=True, sharey="row")

for slurry, ax in zip(unique_slurries, axes):
    ok = X.index.get_level_values("slurry") == slurry
    this_X = X[ok]
    this_phi = y[ok]["phi"].values

    ok_pred = slurries == slurry
    this_Xpred = Xpred[ok_pred]
    this_gpr = phi_gpr[ok_pred.values]

    for ca in this_X["capillary_number"].unique():
        ok = this_X["capillary_number"] == ca
        ok_pred = this_Xpred["capillary_number"] == ca

        ax.scatter(
            this_X[ok]["gap_to_thickness_ratio"],
            this_phi[ok],
            color=cmap(norm(ca)),
        )

        ax.fill_between(
            this_Xpred[ok_pred]["gap_to_thickness_ratio"],
            this_gpr[ok_pred.values]["phi_mean"] - this_gpr[ok_pred.values]["phi_std"],
            this_gpr[ok_pred.values]["phi_mean"] + this_gpr[ok_pred.values]["phi_std"],
            color=cmap(norm(ca)),
            alpha=0.2,
        )

    (cos_theta,) = this_X["cosine_of_contact_angle"].unique()
    ax.set_title(f"Cos θ={cos_theta:.2f}")

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(
    sm, ax=axes, orientation="horizontal", location="top", pad=0.2, aspect=30
)
cbar.set_label("Ca")
quartile_vals = np.quantile(Cas_sorted, [0, 0.25, 0.5, 0.75, 1.0])
nearest_cas = [Cas_sorted[np.argmin(np.abs(Cas_sorted - q))] for q in quartile_vals]
cbar.set_ticks([round(ca, 3) for ca in nearest_cas])
cbar.ax.xaxis.set_major_formatter(ScalarFormatter())

fig.supxlabel("Rgt")
fig.supylabel("phi")
fig.suptitle("Posterior distribution of mean")
plt.show()

## GPQR

In [ ]:
H_gpqr = pd.read_csv("../../_temp/v0/H.quantiles.Xpred_1D.csv")
phi_gpqr = pd.read_csv("../../_temp/v0/phi.quantiles.Xpred_1D.csv")

In [ ]:
fig, axes = plt.subplots(1, len(slurry_map), sharex=True, sharey="row")

for slurry, ax in zip(unique_slurries, axes):
    ok = X.index.get_level_values("slurry") == slurry
    this_X = X[ok]
    this_H = y[ok]["H"].values

    ok_pred = slurries == slurry
    this_Xpred = Xpred[ok_pred]
    this_quantiles = H_gpqr[ok_pred.values]

    for ca in this_X["capillary_number"].unique():
        ok = this_X["capillary_number"] == ca
        ok_pred = this_Xpred["capillary_number"] == ca

        ax.scatter(
            this_X[ok]["gap_to_thickness_ratio"],
            this_H[ok],
            color=cmap(norm(ca)),
        )

        ax.fill_between(
            this_Xpred[ok_pred]["gap_to_thickness_ratio"],
            this_quantiles.iloc[ok_pred.values, 0],
            this_quantiles.iloc[ok_pred.values, -1],
            facecolor=cmap(norm(ca)),
            edgecolor="none",
            alpha=0.3,
        )

    (cos_theta,) = this_X["cosine_of_contact_angle"].unique()
    ax.set_title(f"Cos θ={cos_theta:.2f}")

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(
    sm, ax=axes, orientation="horizontal", location="top", pad=0.2, aspect=30
)
cbar.set_label("Ca")
quartile_vals = np.quantile(Cas_sorted, [0, 0.25, 0.5, 0.75, 1.0])
nearest_cas = [Cas_sorted[np.argmin(np.abs(Cas_sorted - q))] for q in quartile_vals]
cbar.set_ticks([round(ca, 3) for ca in nearest_cas])
cbar.ax.xaxis.set_major_formatter(ScalarFormatter())

fig.supxlabel("Rgt")
fig.supylabel("H")
fig.suptitle("Quantiles")

In [ ]:
fig, axes = plt.subplots(1, len(slurry_map), sharex=True, sharey="row")

for slurry, ax in zip(unique_slurries, axes):
    ok = X.index.get_level_values("slurry") == slurry
    this_X = X[ok]
    this_phi = y[ok]["phi"].values

    ok_pred = slurries == slurry
    this_Xpred = Xpred[ok_pred]
    this_quantiles = phi_gpqr[ok_pred.values]

    for ca in this_X["capillary_number"].unique():
        ok = this_X["capillary_number"] == ca
        ok_pred = this_Xpred["capillary_number"] == ca

        ax.scatter(
            this_X[ok]["gap_to_thickness_ratio"],
            this_phi[ok],
            color=cmap(norm(ca)),
        )

        ax.fill_between(
            this_Xpred[ok_pred]["gap_to_thickness_ratio"],
            this_quantiles.iloc[ok_pred.values, 0],
            this_quantiles.iloc[ok_pred.values, -1],
            facecolor=cmap(norm(ca)),
            edgecolor="none",
            alpha=0.3,
        )

    (cos_theta,) = this_X["cosine_of_contact_angle"].unique()
    ax.set_title(f"Cos θ={cos_theta:.2f}")

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(
    sm, ax=axes, orientation="horizontal", location="top", pad=0.2, aspect=30
)
cbar.set_label("Ca")
quartile_vals = np.quantile(Cas_sorted, [0, 0.25, 0.5, 0.75, 1.0])
nearest_cas = [Cas_sorted[np.argmin(np.abs(Cas_sorted - q))] for q in quartile_vals]
cbar.set_ticks([round(ca, 3) for ca in nearest_cas])
cbar.ax.xaxis.set_major_formatter(ScalarFormatter())

fig.supxlabel("Rgt")
fig.supylabel("phi")
fig.suptitle("Quantiles")